<a href="https://colab.research.google.com/github/Demeter/-/blob/gh-pages/Rebel_Ycombinator_Bullseye.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Bullseye generator - Created by Jared Heyman @ Rebel Fund
# Investor accuracy is probabilistic (expected split) AND center-biased inside inner red:
#   p0       = baseline inner-red hit rate (pure random over outer disk, adjusted for hole size)
#   p_target = p0 + accuracy*(1 - p0)
#   For each shot: with prob p_target -> sample inside INNER RED (center-biased);
#                  else -> sample in the ANNULUS (outer minus inner), uniform-by-area.
# - Draws target (no upload), repeats N times (prompted)
# - Prompts: #holes, hole size, Investor accuracy [0..1], return multiples, #bullseyes
# - Legend shows counts, per-category multiples, weighted avg multiple, investor accuracy
# - Legend auto-placed (never overlaps the outer circle)

import math, random
from PIL import Image, ImageDraw, ImageFilter, ImageFont
from google.colab import files

# ---------- USER INPUTS ----------
def ask_int(prompt, default):
    s = input(f"{prompt} [{default}]: ").strip()
    return int(s) if s else int(default)

def ask_float(prompt, default):
    s = input(f"{prompt} [{default}]: ").strip()
    return float(s) if s else float(default)

def ask_float_01(prompt, default):
    s = input(f"{prompt} (0..1) [{default}]: ").strip()
    v = float(s) if s else float(default)
    return max(0.0, min(1.0, v))

print("Canvas / target")
CANVAS = ask_int("Canvas size (square px)", 1200)

print("\nBullseye areas (as % of OUTER CIRCLE area; must be non-decreasing):")
CENTER_PCT = ask_float("Center disk area % (Decacorns)", 1.0)
INNER_PCT  = ask_float("Inner disk area % (Unicorns, up to red)", 6.0)
MIDDLE_PCT = ask_float("Middle disk area % ($100M+, up to pink)", 20.0)
CENTER_PCT = max(0.0, min(100.0, CENTER_PCT))
INNER_PCT  = max(CENTER_PCT, min(100.0, INNER_PCT))
MIDDLE_PCT = max(INNER_PCT,  min(100.0, MIDDLE_PCT))

print("\nBullets")
N_BULLETS = ask_int("Number of bullet holes", 100)
HOLE_SIZE = ask_int("Hole size (px, same for all)", 22)
INVESTOR_ACCURACY = ask_float_01(
    "Investor accuracy (0=random outer, 1=expected split ~ all inner red; inner hits cluster toward center)", 0.5
)

N_CHARTS = ask_int("How many bullseye charts to generate this run?", 5)

SEED = input("Optional base seed for reproducibility (integer or blank): ").strip()
SEED = int(SEED) if SEED != "" else None

print("\nReturn multiples (defaults shown):")
RM_DECACORN  = ask_float("Decacorns multiple (x)", 500.0)
RM_UNICORN   = ask_float("Unicorns multiple (x)", 50.0)
RM_100M      = ask_float("$100M+ multiple (x)", 10.0)
RM_NONOUT    = ask_float("Non-Outlier multiple (x)", 0.0)

# ---------- COLORS & LABELS ----------
COLR_BG      = (255,255,255,255)
COLR_OUTER   = (233,233,233,255)
COLR_PINK    = (233,170,166,255)
COLR_RED     = (196,67,45,255)
COLR_CENTER  = (138,34,22,255)
COLR_STROKE  = (90,90,90,255)
COLR_TEXT    = (30,30,30,255)

LABEL_DECACORN = "Decacorns"
LABEL_UNICORN  = "Unicorns"
LABEL_100M     = "$100M+"
LABEL_NONOUT   = "Non-Outlier"

# ---------- HELPERS ----------
def gen_hole(size:int):
    S=max(8,size)
    im=Image.new("RGBA",(S,S),(0,0,0,0))
    d=ImageDraw.Draw(im)
    cx,cy=S/2,S/2; r=S*0.38
    pts=[]
    for i in range(14):
        th=2*math.pi*i/14
        rr=r*(1+random.uniform(-.18,.18))
        pts.append((cx+rr*math.cos(th), cy+rr*math.sin(th)))
    d.polygon(pts,fill=(40,40,40,255))
    d.ellipse((cx-r*.45,cy-r*.45,cx+r*.45,cy+r*.45),fill=(12,12,12,255))
    hl=Image.new("L",(S,S),0); ImageDraw.Draw(hl).ellipse((cx-r*.9,cy-r*.9,cx+r*.9,cy+r*.9),fill=180)
    hl=hl.filter(ImageFilter.GaussianBlur(radius=S*.05))
    rim=Image.new("RGBA",(S,S),(255,255,255,0)); rim.putalpha(hl.point(lambda a:int(a*.25)))
    return Image.alpha_composite(im,rim)

def sample_uniform_disk(cx, cy, R):
    u = random.random()
    r = math.sqrt(u) * R
    th = random.uniform(0, 2*math.pi)
    return cx + r*math.cos(th), cy + r*math.sin(th)

def sample_uniform_annulus(cx, cy, r_in, r_out):
    # area-uniform between radii r_in..r_out
    u = random.random()
    r = math.sqrt((r_out*r_out - r_in*r_in) * u + r_in*r_in)
    th = random.uniform(0, 2*math.pi)
    return cx + r*math.cos(th), cy + r*math.sin(th)

def sample_center_biased_disk(cx, cy, R, strength_01):
    """
    Area-correct radial bias toward center inside a disk.
    strength_01 in [0,1]: 0 -> uniform-by-area, 1 -> strong pull.
    Implemented via r = R * U**beta, where beta = 0.5 + 2*strength.
    """
    beta = 0.5 + 2.0*max(0.0, min(1.0, strength_01))
    u = random.random()
    r = (u ** beta) * R
    th = random.uniform(0, 2*math.pi)
    return cx + r*math.cos(th), cy + r*math.sin(th)

def ellipse_bbox(cx,cy,r): return (cx-r,cy-r,cx+r,cy+r)

def load_font(sz=28):
    try: return ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", sz)
    except: return ImageFont.load_default()

def measure_block(draw, lines, font, pad=12):
    max_w, total_h = 0, 0
    for line in lines:
        bbox = draw.textbbox((0,0), line, font=font)
        w = bbox[2]-bbox[0]; h = bbox[3]-bbox[1]
        max_w = max(max_w, w); total_h += h + 6
    total_h -= 6
    return max_w + 2*pad, total_h + 2*pad

def draw_block(draw, x, y, lines, font, fill, pad=12, bg=(255,255,255,235)):
    w, h = measure_block(draw, lines, font, pad)
    draw.rounded_rectangle([x, y, x+w, y+h], radius=12, fill=bg, outline=(200,200,200,255), width=2)
    cy = y + pad
    for line in lines:
        draw.text((x+pad, cy), line, font=font, fill=fill)
        bbox = draw.textbbox((0,0), line, font=font)
        cy += (bbox[3]-bbox[1]) + 6
    return w, h

def place_legend(out, lines, cx, cy, R, COLR_TEXT, margin_frac=0.04):
    d2 = ImageDraw.Draw(out)
    font_leg = load_font(28)
    pad = 14
    block_w, block_h = measure_block(d2, lines, font_leg, pad)
    W, H = out.size
    margin = int(margin_frac * W)
    candidates = [
        (int(cx + R + margin),          int(cy - block_h/2)),            # right
        (int(cx - R - margin - block_w),int(cy - block_h/2)),            # left
        (int(cx - block_w/2),           int(cy + R + margin)),           # below
        (int(cx - block_w/2),           int(cy - R - margin - block_h)), # above
    ]
    def fits(x,y):
        return x>=margin and y>=margin and x+block_w<=W-margin and y+block_h<=H-margin
    pos=None
    for (lx,ly) in candidates:
        if fits(lx,ly):
            pos=(lx,ly); break
    if pos is None:
        pos=(min(max(margin,cx+R+margin),W-margin-block_w),
             min(max(margin,cy-block_h/2),H-margin-block_h))
    draw_block(d2, pos[0], pos[1], lines, font_leg, COLR_TEXT, pad)

# ---------- CORE GENERATION (single chart) ----------
def make_one_chart(chart_idx: int):
    if SEED is not None:
        random.seed(SEED + chart_idx)

    # 1) draw target
    W=H=CANVAS
    cx,cy=W/2,H/2
    R=0.48*min(W,H)  # canvas margin

    def r_for_pct(p): return R*math.sqrt(max(0.0,min(100.0,p))/100.0)
    r_c = r_for_pct(CENTER_PCT)   # Decacorns (center dark)
    r_r = r_for_pct(INNER_PCT)    # Unicorns (inner red)
    r_p = r_for_pct(MIDDLE_PCT)   # $100M+ (pink)

    img=Image.new("RGBA",(W,H),COLR_BG)
    d=ImageDraw.Draw(img)
    d.ellipse(ellipse_bbox(cx,cy,R), fill=COLR_OUTER, outline=COLR_STROKE, width=2)
    d.ellipse(ellipse_bbox(cx,cy,r_p), fill=COLR_PINK)
    d.ellipse(ellipse_bbox(cx,cy,r_r), fill=COLR_RED)
    d.ellipse(ellipse_bbox(cx,cy,r_c), fill=COLR_CENTER)

    # 2) place bullets (non-overlap, fully inside)
    hole = gen_hole(HOLE_SIZE)
    hr   = hole.width/2

    usable_outer = R   - hr - 1          # outer usable radius
    usable_inner = r_r - hr - 1          # inner RED usable radius

    if usable_outer <= 2:
        raise SystemExit("Outer circle too small for this HOLE_SIZE. Reduce size or increase canvas.")

    inner_enabled = usable_inner > 2

    # Baseline random probability p0 (using usable areas)
    if inner_enabled:
        p0 = (usable_inner*usable_inner) / (usable_outer*usable_outer)
    else:
        p0 = 0.0

    # Target expected inner probability (probabilistic accuracy)
    p_target = p0 + INVESTOR_ACCURACY * (1.0 - p0)
    p_target = max(0.0, min(1.0, p_target))
    if not inner_enabled and p_target > 0:
        print("⚠️ Inner red is too small for this HOLE_SIZE; falling back to outer-only sampling.")
        p_target = 0.0

    # Inside-inner center bias strength (how clustered near the actual center)
    inner_bias_strength = INVESTOR_ACCURACY  # 0 -> uniform, 1 -> strong pull

    cell = int(math.ceil(2*hr))
    grid = {}
    pts  = []
    MAX_TRIES = 300000

    def idx(x,y): return int(x//cell), int(y//cell)
    def overlap(x,y):
        ix,iy = idx(x,y)
        for dx in (-1,0,1):
            for dy in (-1,0,1):
                for j in grid.get((ix+dx,iy+dy), []):
                    px,py = pts[j]
                    if (x-px)**2 + (y-py)**2 < (2*hr)**2:
                        return True
        return False

    tries=0
    while len(pts) < N_BULLETS and tries < MAX_TRIES:
        tries += 1
        if random.random() < p_target:
            # inner red (center-biased)
            x,y = sample_center_biased_disk(cx, cy, usable_inner, inner_bias_strength)
        else:
            # annulus: outer minus inner (uniform by area)
            r_low = usable_inner if inner_enabled else 0.0
            x,y = sample_uniform_annulus(cx, cy, r_low, usable_outer)

        if overlap(x,y):
            continue
        pts.append((x,y))
        ix,iy = idx(x,y)
        grid.setdefault((ix,iy), []).append(len(pts)-1)

    out = img.copy()
    for (x,y) in pts:
        out.alpha_composite(hole, (int(x-hr), int(y-hr)))

    # 3) counts & ROI
    def counts(pts,cx,cy,r_c,r_r,r_p,R):
        a=b=c=d_=0
        for (x,y) in pts:
            dist = math.hypot(x-cx, y-cy)
            if   dist <= r_c: a+=1
            elif dist <= r_r: b+=1
            elif dist <= r_p: c+=1
            elif dist <= R  : d_+=1
        return a,b,c,d_
    c_dec,c_uni,c_100,c_non = counts(pts,cx,cy,r_c,r_r,r_p,R)
    tot = len(pts)
    w_sum = c_dec*RM_DECACORN + c_uni*RM_UNICORN + c_100*RM_100M + c_non*RM_NONOUT
    w_avg = w_sum / tot if tot else 0.0

    # 4) legend
    legend_lines = [
        f"Shots: {tot}",
        f"Investor accuracy: {INVESTOR_ACCURACY:.2f}",
        f"{LABEL_DECACORN}: {c_dec}  @ {RM_DECACORN:.0f}x",
        f"{LABEL_UNICORN}:  {c_uni}  @ {RM_UNICORN:.0f}x",
        f"{LABEL_100M}:     {c_100}  @ {RM_100M:.0f}x",
        f"{LABEL_NONOUT}:   {c_non}  @ {RM_NONOUT:.0f}x",
        f"Weighted avg multiple: {w_avg:.2f}x",
    ]
    place_legend(out, legend_lines, cx, cy, R, COLR_TEXT)

    # 5) save one chart
    name=f"bullseye_{chart_idx}_{N_BULLETS}holes_{HOLE_SIZE}px_acc{INVESTOR_ACCURACY:.2f}.png"
    out.save(name,"PNG")
    print(f"✅ Saved chart {chart_idx}: {name}")
    return name

# ---------- MAKE N CHARTS ----------
saved_files = []
for i in range(1, N_CHARTS + 1):
    if SEED is not None:
        random.seed(SEED + i)
    saved_files.append(make_one_chart(i))

# ---------- OFFER DOWNLOADS ----------
print("All files saved:")
for f in saved_files:
    print(" •", f)
for f in saved_files:
    files.download(f)

Canvas / target
Canvas size (square px) [1200]: 

Bullseye areas (as % of OUTER CIRCLE area; must be non-decreasing):
Center disk area % (Decacorns) [1.0]: 
Inner disk area % (Unicorns, up to red) [6.0]: 
Middle disk area % ($100M+, up to pink) [20.0]: 

Bullets
Number of bullet holes [100]: 150
Hole size (px, same for all) [22]: 
Investor accuracy (0=random outer, 1=expected split ~ all inner red; inner hits cluster toward center) (0..1) [0.5]: 0.1
How many bullseye charts to generate this run? [5]: 
Optional base seed for reproducibility (integer or blank): 

Return multiples (defaults shown):
Decacorns multiple (x) [500.0]: 
Unicorns multiple (x) [50.0]: 
$100M+ multiple (x) [10.0]: 
Non-Outlier multiple (x) [0.0]: 
✅ Saved chart 1: bullseye_1_150holes_22px_acc0.10.png
✅ Saved chart 2: bullseye_2_150holes_22px_acc0.10.png
✅ Saved chart 3: bullseye_3_150holes_22px_acc0.10.png
✅ Saved chart 4: bullseye_4_150holes_22px_acc0.10.png
✅ Saved chart 5: bullseye_5_150holes_22px_acc0.10.png
A

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>